# Kepler-refine vs N-body — error characterisation

Companion notebook to [`docs/kepler_refine_error_report.md`](../docs/kepler_refine_error_report.md).

Inputs:
- `data/output/kepler_vs_nbody_comparison.parquet` (796 pairs)

Outputs (figures): displayed inline.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 110

df = pl.read_parquet('../data/output/kepler_vs_nbody_comparison.parquet')
ok = df.filter(pl.col('error_message').is_null()).with_columns(
    pl.col('delta_dist_au').abs().alias('abs_dd'),
    pl.col('delta_t_min_hours').abs().alias('abs_dt'),
    pl.max_horizontal('e_1','e_2').alias('e_max'),
    pl.max_horizontal('i_1','i_2').alias('i_max'),
    pl.min_horizontal('q_1','q_2').alias('q_min'),
)
print(f'rows: {len(ok)}')
print(f'near_boundary: {int(ok.filter(pl.col("near_boundary")).height)}')
print()
for col in ['abs_dd', 'abs_dt']:
    vals = ok[col].to_numpy()
    print(f'{col}: med={np.median(vals):.3e}  p95={np.quantile(vals, 0.95):.3e}  p99={np.quantile(vals, 0.99):.3e}  max={vals.max():.3e}')

## Global histograms (log-scale)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.hist(np.log10(ok['abs_dd'].to_numpy().clip(1e-9, None)), bins=50, color='C0', alpha=0.85)
for thr, label in [(1e-4, '0.1 mAU'), (1e-3, '1 mAU'), (1e-2, '10 mAU')]:
    ax.axvline(np.log10(thr), ls='--', color='k', alpha=0.4)
    ax.text(np.log10(thr), ax.get_ylim()[1]*0.95, ' '+label, va='top', fontsize=8)
ax.set_xlabel('log10 |Δdist_au|')
ax.set_ylabel('count')
ax.set_title('Distance disagreement (Kepler − N-body)')

ax = axes[1]
ax.hist(ok['abs_dt'].to_numpy(), bins=40, color='C1', alpha=0.85)
ax.set_xlabel('|Δt_min| (h)')
ax.set_ylabel('count')
ax.set_title('Epoch disagreement; clip at 12h boundary')
ax.axvline(12, ls='--', color='r', alpha=0.5)

plt.tight_layout()
plt.show()

## Error vs orbital parameters

Boxplots binned by e_max and q_min.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

def boxplot_by(ax, column, edges, labels, title):
    vals = ok[column].to_numpy()
    dd = ok['abs_dd'].to_numpy()
    data = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (vals >= lo) & (vals < hi)
        data.append(dd[mask])
    ax.boxplot(data, labels=labels, showfliers=False)
    ax.set_yscale('log')
    ax.set_ylabel('|Δdist_au|')
    ax.set_xlabel(column)
    ax.set_title(title)
    ax.grid(True, alpha=0.3, which='both')

boxplot_by(axes[0], 'e_max',
           [0, 0.10, 0.20, 0.30, 0.45, 0.70],
           ['<0.10','0.10-0.20','0.20-0.30','0.30-0.45','0.45-0.70'],
           '|Δdist| vs e_max')
boxplot_by(axes[1], 'q_min',
           [0, 1.3, 1.8, 2.2, 2.6, 3.0, 5.0],
           ['<1.3','1.3-1.8','1.8-2.2','2.2-2.6','2.6-3.0','>3.0'],
           '|Δdist| vs q_min')
boxplot_by(axes[2], 'i_max',
           [0, 3, 6, 10, 15, 25, 40],
           ['<3','3-6','6-10','10-15','15-25','>25'],
           '|Δdist| vs i_max')

plt.tight_layout()
plt.show()

## (e_max, q_min) heatmap of p95 |Δdist|

In [ ]:
e_edges = np.array([0, 0.10, 0.20, 0.30, 0.45, 0.70])
q_edges = np.array([0, 1.3, 1.8, 2.2, 2.6, 3.0, 5.0])

e = ok['e_max'].to_numpy()
q = ok['q_min'].to_numpy()
dd = ok['abs_dd'].to_numpy()

grid_p95 = np.full((len(e_edges)-1, len(q_edges)-1), np.nan)
grid_n   = np.zeros_like(grid_p95)

for i, (e_lo, e_hi) in enumerate(zip(e_edges[:-1], e_edges[1:])):
    for j, (q_lo, q_hi) in enumerate(zip(q_edges[:-1], q_edges[1:])):
        mask = (e >= e_lo) & (e < e_hi) & (q >= q_lo) & (q < q_hi)
        if mask.sum() >= 3:
            grid_p95[i, j] = np.quantile(dd[mask], 0.95)
        grid_n[i, j] = mask.sum()

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(np.log10(grid_p95), origin='lower', aspect='auto', cmap='viridis')
ax.set_xticks(range(len(q_edges)-1))
ax.set_xticklabels([f'{lo}-{hi}' for lo, hi in zip(q_edges[:-1], q_edges[1:])])
ax.set_yticks(range(len(e_edges)-1))
ax.set_yticklabels([f'{lo}-{hi}' for lo, hi in zip(e_edges[:-1], e_edges[1:])])
ax.set_xlabel('q_min (AU)')
ax.set_ylabel('e_max')
ax.set_title('log10 p95(|Δdist_au|) by (e_max, q_min) bin')
for i in range(grid_p95.shape[0]):
    for j in range(grid_p95.shape[1]):
        n = int(grid_n[i, j])
        if n >= 3:
            ax.text(j, i, f'n={n}', ha='center', va='center', fontsize=8, color='white')
plt.colorbar(im, ax=ax, label='log10 p95(|Δdist_au|)')
plt.tight_layout()
plt.show()

## Top-20 worst pairs

In [ ]:
top = ok.sort('abs_dd', descending=True).head(20).select([
    'number_1', 'number_2',
    'dist_au_kepler', 'dist_au_nbody', 'abs_dd',
    'delta_t_min_hours', 'e_max', 'i_max', 'q_min', 'near_boundary',
])
top.to_pandas()